
# 01 — Fundamentos Modernos de PLN y LLMs
## De Sistemas Basados en Reglas a Modelos de Lenguaje Modernos

---

# Descripción del Notebook

Este notebook introduce el ecosistema moderno de Procesamiento del Lenguaje Natural (PLN) mediante experimentación práctica y ejercicios orientados a ingeniería de IA.

Durante este laboratorio los estudiantes:

- Explorarán cómo las computadoras procesan lenguaje humano
- Compararán NLP tradicional vs transformers modernos
- Generarán embeddings semánticos
- Experimentarán con modelos de HuggingFace
- Analizarán limitaciones y alucinaciones
- Comprenderán la evolución desde sistemas basados en reglas hasta LLMs modernos

---

# Duración Estimada

⏱ 4–6 horas dependiendo del nivel de experimentación.

---

# Objetivos de Aprendizaje

Al finalizar este notebook el estudiante podrá:

- Comprender qué es PLN y por qué el lenguaje es difícil para las computadoras
- Identificar cuándo utilizar IA conversacional y cuándo evitarla
- Comparar sistemas basados en reglas, Machine Learning, Deep Learning, Transformers y LLMs
- Construir pipelines básicos de NLP
- Generar y comparar embeddings semánticos
- Experimentar con modelos transformer modernos
- Analizar limitaciones y alucinaciones
- Pensar críticamente como ingeniero de IA



# ============================================================
# SECCIÓN 1 — Bienvenida a IA II
# ============================================================

# Bienvenidos al curso

Este curso está enfocado en:

- NLP moderno
- LLMs
- Embeddings
- RAG
- Recuperación semántica
- Asistentes conversacionales
- Ingeniería de IA

La idea NO es memorizar definiciones.

La idea es:
- experimentar,
- construir,
- analizar,
- depurar,
- evaluar,
- y pensar críticamente sobre sistemas modernos de IA.


In [ ]:

# ============================================================
# INSTALACIÓN DE LIBRERÍAS
# ============================================================

!pip -q install transformers
!pip -q install sentence-transformers
!pip -q install scikit-learn
!pip -q install datasets
!pip -q install accelerate
!pip -q install umap-learn


In [ ]:

# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import pipeline

import warnings
warnings.filterwarnings("ignore")


In [ ]:

# ============================================================
# VERIFICAR GPU
# ============================================================

import torch

print("GPU Disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



# ============================================================
# SECCIÓN 2 — ¿Por Qué el Lenguaje Humano es Difícil?
# ============================================================

Los humanos entendemos:
- sarcasmo,
- contexto,
- ambigüedad,
- emociones,
- expresiones culturales,
- dobles sentidos.

Las computadoras no.

Ese es uno de los principales retos históricos del PLN.


In [ ]:

# ============================================================
# EJEMPLOS DE AMBIGÜEDAD
# ============================================================

frases = [
    "Qué buena nota",
    "Estoy muriéndome de hambre",
    "Buenísimo... se cayó el sistema otra vez",
    "Mae qué carga",
    "Excelente... perdimos todos los datos"
]

for i, frase in enumerate(frases):
    print(f"{i+1}. {frase}")



# Actividad de Discusión

Para cada frase:

1. ¿Cuál es el significado literal?
2. ¿Cuál es el significado real?
3. ¿Podría una máquina confundirse?
4. ¿Qué información contextual ayuda a los humanos?



# ============================================================
# SECCIÓN 3 — ¿Qué Problemas Resuelve el PLN?
# ============================================================

Actualmente usamos PLN constantemente:

- Chatbots
- Siri y Alexa
- Traducción automática
- Análisis de sentimientos
- Recuperación inteligente de información
- Clasificación de texto
- Filtros de spam
- IA generativa


In [ ]:

# ============================================================
# EJEMPLO SIMPLE DE CLASIFICACIÓN
# ============================================================

correos = [
    "Gana dinero ahora",
    "Reunión mañana a las 3",
    "Oferta limitada",
    "Actualización del proyecto",
    "Felicidades ganaste un premio",
    "Notas de la reunión"
]

etiquetas = [
    "spam",
    "normal",
    "spam",
    "normal",
    "spam",
    "normal"
]

df = pd.DataFrame({
    "correo": correos,
    "etiqueta": etiquetas
})

df



# Observación de Ingeniería

Incluso este pequeño dataset representa un problema real de NLP:

¿Cómo puede una computadora clasificar automáticamente texto?



# ============================================================
# SECCIÓN 4 — ¿Cuándo NO Usar IA?
# ============================================================

La IA NO siempre es la mejor solución.

En algunos casos:
- reglas simples,
- scripts,
- consultas SQL,
- automatizaciones tradicionales,

son mejores que utilizar LLMs.

Pensar como ingeniero significa también saber cuándo NO usar IA.



# Actividad

Analice los siguientes escenarios:

| Escenario | ¿Usar IA? | ¿Por Qué? |
|---|---|---|
| Diagnóstico médico crítico | ? | ? |
| Chatbot FAQ | ? | ? |
| Control nuclear | ? | ? |
| Análisis de sentimientos | ? | ? |
| Filtro de spam | ? | ? |
| Decisiones legales automáticas | ? | ? |



# ============================================================
# SECCIÓN 5 — NLP Tradicional: Bag of Words
# ============================================================

Antes de transformers y LLMs, NLP dependía fuertemente de:

- reglas,
- estadísticas,
- ingeniería manual de características.

Uno de los enfoques más comunes era:
# Bag of Words


In [ ]:

# ============================================================
# BAG OF WORDS
# ============================================================

documentos = [
    "Me gustan los perros",
    "Los perros son increíbles",
    "Me gusta la pizza"
]

vectorizador = CountVectorizer()

X = vectorizador.fit_transform(documentos)

print(vectorizador.get_feature_names_out())


In [ ]:

# REPRESENTACIÓN NUMÉRICA

X.toarray()



# ¿Qué acaba de ocurrir?

El texto fue convertido a números.

Los modelos de Machine Learning NO entienden texto directamente.

Pero Bag of Words tiene limitaciones importantes:
- no entiende contexto,
- no entiende significado,
- ignora el orden de palabras,
- no comprende relaciones semánticas.


In [ ]:

# ============================================================
# PROBLEMA DE RELACIONES SEMÁNTICAS
# ============================================================

documentos = [
    "Me encantan los perros",
    "Los perros son animales maravillosos",
    "Quiero comer pizza"
]

vectorizador = CountVectorizer()

X = vectorizador.fit_transform(documentos)

pd.DataFrame(
    X.toarray(),
    columns=vectorizador.get_feature_names_out()
)



# Discusión

Observe que:

- "Me encantan los perros"
- "Los perros son animales maravillosos"

son frases relacionadas semánticamente.

Sin embargo Bag of Words NO comprende realmente esa relación.



# ============================================================
# SECCIÓN 6 — TF-IDF
# ============================================================

TF-IDF mejora Bag of Words asignando importancia diferente a las palabras.

Palabras muy frecuentes en TODOS los documentos reciben menor peso.


In [ ]:

# ============================================================
# TF-IDF
# ============================================================

tfidf = TfidfVectorizer()

X_tfidf = tfidf.fit_transform(documentos)

pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf.get_feature_names_out()
)



# ============================================================
# SECCIÓN 7 — Pipeline Básico de Machine Learning
# ============================================================

Ahora construiremos un clasificador sencillo de spam utilizando:
- TF-IDF
- Naive Bayes


In [ ]:

# ============================================================
# DATASET
# ============================================================

mensajes = [
    "Gana dinero ahora",
    "Reclama tu premio",
    "Reunión mañana",
    "Actualización del proyecto",
    "Vacaciones gratis",
    "Almuerzo de trabajo",
    "Ganaste un automóvil",
    "Presentación al cliente"
]

labels = [
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    0
]


In [ ]:

# DIVISIÓN DE DATOS

X_train, X_test, y_train, y_test = train_test_split(
    mensajes,
    labels,
    test_size=0.25,
    random_state=42
)


In [ ]:

# VECTORIZACIÓN

vectorizer = TfidfVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


In [ ]:

# ENTRENAMIENTO DEL MODELO

modelo = MultinomialNB()

modelo.fit(X_train_vec, y_train)


In [ ]:

# PREDICCIONES

predicciones = modelo.predict(X_test_vec)

predicciones


In [ ]:

# EVALUACIÓN

accuracy = accuracy_score(y_test, predicciones)

print("Accuracy:", accuracy)


In [ ]:

# MATRIZ DE CONFUSIÓN

cm = confusion_matrix(y_test, predicciones)

sns.heatmap(cm, annot=True, fmt="d")

plt.title("Matriz de Confusión")
plt.xlabel("Predicción")
plt.ylabel("Real")

plt.show()



# Nota de Ingeniería

Los pipelines tradicionales:
- son rápidos,
- ligeros,
- relativamente interpretables.

Y todavía se utilizan actualmente en muchos sistemas reales.



# ============================================================
# SECCIÓN 8 — Deep Learning y Embeddings
# ============================================================

Deep Learning revolucionó NLP porque los modelos comenzaron a aprender representaciones complejas automáticamente.

Uno de los conceptos más importantes:
# Embeddings


In [ ]:

# ============================================================
# CARGAR MODELO DE EMBEDDINGS
# ============================================================

embedding_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)


In [ ]:

# FRASES DE EJEMPLO

frases = [
    "Me gustan los perros",
    "Los perros son animales increíbles",
    "La inteligencia artificial avanza rápidamente",
    "La pizza sabe deliciosa"
]


In [ ]:

# GENERAR EMBEDDINGS

embeddings = embedding_model.encode(frases)

print(embeddings.shape)



# Observación

Cada frase ahora se representa como un vector numérico denso.

A diferencia de Bag of Words:
- sí existe comprensión semántica,
- se preserva contexto,
- se capturan relaciones complejas.


In [ ]:

# ============================================================
# SIMILITUD COSENO
# ============================================================

similarity_matrix = cosine_similarity(embeddings)

similarity_matrix


In [ ]:

# VISUALIZACIÓN DE SIMILITUD

sns.heatmap(
    similarity_matrix,
    annot=True,
    xticklabels=frases,
    yticklabels=frases
)

plt.title("Matriz de Similitud")

plt.show()



# Análisis

Observe cómo las frases relacionadas con perros son más similares entre sí.

Esto es la base de:
- búsqueda semántica,
- RAG,
- sistemas de recomendación,
- bases vectoriales.



# ============================================================
# SECCIÓN 9 — Transformers
# ============================================================

Transformers revolucionaron NLP en 2017.

La innovación principal:
# Self-Attention

Los transformers permiten:
- analizar relaciones entre palabras,
- procesar texto en paralelo,
- comprender contexto de largo alcance,
- escalar a modelos gigantes.



# Nota de Ingeniería

La mayoría de sistemas modernos de IA utilizan transformers:

- GPT
- BERT
- Gemini
- Claude
- LLaMA
- Mistral



# ============================================================
# SECCIÓN 10 — Pipelines Modernos con HuggingFace
# ============================================================


In [ ]:

# ============================================================
# ANÁLISIS DE SENTIMIENTOS
# ============================================================

sentiment_pipeline = pipeline(
    "sentiment-analysis"
)

resultados = sentiment_pipeline([
    "Me encanta este curso",
    "Este sistema es terrible",
    "La película estuvo regular"
])

resultados


In [ ]:

# ============================================================
# RESUMEN AUTOMÁTICO
# ============================================================

summarizer = pipeline(
    "summarization"
)

texto = '''
La inteligencia artificial está transformando las industrias modernas.
Las empresas integran IA en salud, finanzas, educación, logística y desarrollo de software.
'''

summary = summarizer(
    texto,
    max_length=40,
    min_length=10,
    do_sample=False
)

summary


In [ ]:

# ============================================================
# QUESTION ANSWERING
# ============================================================

qa_pipeline = pipeline(
    "question-answering"
)

contexto = '''
Los transformers fueron introducidos en 2017 y cambiaron completamente NLP moderno.
'''

pregunta = "¿Cuándo fueron introducidos los transformers?"

qa_pipeline(
    question=pregunta,
    context=contexto
)



# ============================================================
# SECCIÓN 11 — Alucinaciones y Errores
# ============================================================

Los LLMs pueden:
- inventar información,
- generar datos falsos,
- responder incorrectamente con mucha confianza.

Esto se conoce como:
# Hallucinations



# Actividad Experimental

Pruebe prompts:
- ambiguos,
- contradictorios,
- falsos,
- o imposibles.

Analice:
- errores,
- confianza,
- sesgos,
- inconsistencias.



# Preguntas de Reflexión

1. ¿Por qué ocurren las alucinaciones?
2. ¿Por qué son peligrosas?
3. ¿Qué industrias se ven más afectadas?
4. ¿Se pueden eliminar completamente?



# ============================================================
# SECCIÓN 12 — Ecosistema Moderno de LLMs
# ============================================================

Actualmente existen múltiples modelos modernos:

- GPT
- Claude
- Gemini
- LLaMA
- Mistral

Cada uno difiere en:
- costo,
- velocidad,
- contexto,
- privacidad,
- razonamiento,
- apertura.



# Actividad

Realice la misma pregunta en:
- ChatGPT
- Gemini
- Claude

Compare:
- calidad,
- razonamiento,
- alucinaciones,
- estilo,
- precisión.



# ============================================================
# SECCIÓN 13 — Mini Proyecto
# ============================================================

# Mini Proyecto — Buscador Semántico

Objetivo:
Construir un pequeño buscador semántico utilizando embeddings.


In [ ]:

# DOCUMENTOS

documentos = [
    "La inteligencia artificial está transformando la medicina",
    "El fútbol es el deporte más popular",
    "Los transformers revolucionaron NLP",
    "Deep Learning requiere grandes datasets",
    "Las recetas de pizza son populares"
]


In [ ]:

# GENERAR EMBEDDINGS

doc_embeddings = embedding_model.encode(documentos)


In [ ]:

# CONSULTA DEL USUARIO

query = "¿Cómo cambiaron los transformers los modelos de lenguaje?"


In [ ]:

# EMBEDDING DE LA CONSULTA

query_embedding = embedding_model.encode([query])


In [ ]:

# SIMILITUDES

scores = cosine_similarity(
    query_embedding,
    doc_embeddings
)

scores


In [ ]:

# MEJOR RESULTADO

best_index = np.argmax(scores)

print("Consulta:")
print(query)

print("\nMejor Resultado:")
print(documentos[best_index])



# Reflexión de Ingeniería

Esto representa la base de:
- RAG,
- recuperación semántica,
- búsqueda vectorial,
- asistentes empresariales.



# ============================================================
# SECCIÓN 14 — Reflexión Final
# ============================================================

El PLN evolucionó desde:
- sistemas basados en reglas,
- Machine Learning tradicional,
- Deep Learning,
- Transformers,
- hasta LLMs modernos.

La IA moderna es extremadamente poderosa.

Pero también introduce:
- sesgos,
- riesgos éticos,
- alucinaciones,
- problemas de privacidad,
- retos operativos.

Comprender estas limitaciones es fundamental para cualquier ingeniero de IA moderno.



# ============================================================
# RETOS OPCIONALES
# ============================================================

1. Compare dos modelos de embeddings
2. Construya un buscador semántico multilenguaje
3. Analice sarcasmo
4. Cree su propio dataset de spam
5. Compare TF-IDF vs embeddings
6. Pruebe diferentes pipelines de HuggingFace
7. Visualice embeddings usando PCA o UMAP
